# Лабораторная работа №2 Обработка пропусков в данных, кодирование категориальных признаков, масштабирование данных.

Панасюк Ксения ИУ5-64Б

Цель лабораторной работы: изучение способов предварительной обработки данных для дальнейшего формирования моделей.

Задание: Выбрать набор данных (датасет), содержащий категориальные признаки и пропуски в данных. Для выполнения следующих пунктов можно использовать несколько различных наборов данных (один для обработки пропусков, другой для категориальных признаков и т.д.)

Для выбранного датасета (датасетов) на основе материалов лекции решить следующие задачи

        обработку пропусков в данных;
        кодирование категориальных признаков;
        масштабирование данных.

# 1) Импорты

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler


# 2) Загрузка данных
Используем датасет "Ames Housing dataset" (цены на жилье в Эймсе)

In [18]:
housing = fetch_openml(name="house_prices", as_frame=True, parser='pandas')
data = housing.frame

print(f"Размер датасета: {data.shape}")

Размер датасета: (1460, 81)


In [19]:
cols_to_use = [
    'LotFrontage', 'LotArea', 'Street', 'MasVnrType', 'MasVnrArea', 
    'ExterQual', 'Foundation', 'FullBath', 'BedroomAbvGr', 'SalePrice', 'Electrical', 'GarageArea'
]

df = data[cols_to_use].copy()
display(df.head())

,LotFrontage,LotArea,Street,MasVnrType,MasVnrArea,ExterQual,Foundation,FullBath,BedroomAbvGr,SalePrice,Electrical,GarageArea
0,65.0,8450,Pave,BrkFace,196.0,Gd,PConc,2,3,208500,SBrkr,548
1,80.0,9600,Pave,None,0.0,TA,CBlock,2,3,181500,SBrkr,460
2,68.0,11250,Pave,BrkFace,162.0,Gd,PConc,2,3,223500,SBrkr,608
3,60.0,9550,Pave,None,0.0,TA,BrkTil,1,3,140000,SBrkr,642
4,84.0,14260,Pave,BrkFace,350.0,Gd,PConc,2,4,250000,SBrkr,836


# 3) Задача 1: Обработка пропусков в данных

In [20]:
print("Пропуски до обработки:")
print(df[['LotFrontage', 'Electrical', 'MasVnrArea']].isnull().sum())

# 1. Числовой: LotFrontage(Линейный размер фасада участка) (Медиана)
imputer_median = SimpleImputer(strategy='median')
df['LotFrontage'] = imputer_median.fit_transform(df[['LotFrontage']]).ravel()

# 2. Категориальный: Electrical(Тип электричества) (Мода)
df['Electrical'] = df['Electrical'].astype(object)
imputer_mode = SimpleImputer(strategy='most_frequent')
df['Electrical'] = imputer_mode.fit_transform(df[['Electrical']]).ravel()

# 3. Числовой: MasVnrArea (Площадь облицовки (Заполнение нулем)
# Логика: если площадь облицовки не указана, скорее всего, облицовки нет (площадь = 0)
df['MasVnrArea'] = df['MasVnrArea'].fillna(0)

print("\nПропуски после обработки:")
print(df[['LotFrontage', 'Electrical', 'MasVnrArea']].isnull().sum())

Пропуски до обработки:
LotFrontage    259
Electrical       1
MasVnrArea       8
dtype: int64

Пропуски после обработки:
LotFrontage    0
Electrical     0
MasVnrArea     0
dtype: int64


# 4) Задача 2: Кодирование категориальных признаков

In [21]:
# Пример One-Hot Encoding для 'Foundation' (тип фундамента)
# Используем pandas get_dummies для простоты визуализации в отчете
df = pd.get_dummies(df, columns=['Foundation'], prefix='Found')

# Пример порядкового кодирования (Ordinal Encoding) для 'ExterQual' (качество отделки)
# Здесь важен порядок: Ex (Excellent) > Gd (Good) > TA (Average) > Fa (Fair)
qual_map = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0}
df['ExterQual_coded'] = df['ExterQual'].map(qual_map)

print("Данные после кодирования (новые колонки):")
display(df[['ExterQual', 'ExterQual_coded'] + [c for c in df.columns if 'Found_' in c]].head())

Данные после кодирования (новые колонки):


,ExterQual,ExterQual_coded,Found_BrkTil,Found_CBlock,Found_PConc,Found_Slab,Found_Stone,Found_Wood
0,Gd,3,False,False,True,False,False,False
1,TA,2,False,True,False,False,False,False
2,Gd,3,False,False,True,False,False,False
3,TA,2,True,False,False,False,False,False
4,Gd,3,False,False,True,False,False,False


# 5) Задача 3: Масштабирование данных

In [22]:
# Масштабируем площадь участка (LotArea) и цену (SalePrice)
# StandardScaler для LotArea
std_scaler = StandardScaler()
df['LotArea_std'] = std_scaler.fit_transform(df[['LotArea']])

# MinMaxScaler для SalePrice
mm_scaler = MinMaxScaler()
df['SalePrice_norm'] = mm_scaler.fit_transform(df[['SalePrice']])

print("Результат масштабирования:")
display(df[['LotArea', 'LotArea_std', 'SalePrice', 'SalePrice_norm']].head())

Результат масштабирования:


,LotArea,LotArea_std,SalePrice,SalePrice_norm
0,8450,-0.207142,208500,0.241078
1,9600,-0.091886,181500,0.203583
2,11250,0.073480,223500,0.261908
3,9550,-0.096897,140000,0.145952
4,14260,0.375148,250000,0.298709
